In [1]:
import torch
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
import warnings
warnings.filterwarnings("ignore")

CONFIG = {
    "model_name": "facebook/mbart-large-50-many-to-many-mmt",
    "source_lang": "en_XX",
    "target_lang": "bn_IN",
    "max_sentence_length": 128,
    "num_beams": 5,
    "repetition_penalty": 1.2,
    "length_penalty": 1.0,
    "temperature": 0.7,
    "do_sample": True,
    "top_p": 0.9,
    "max_new_tokens": 128
}

class EnglishToBengaliTranslator:
    def __init__(self, model_name=None):
        """
        Initialize the translator with pre-trained mBART model
        """
        self.model_name = model_name or CONFIG["model_name"]
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        print("Loading pre-trained mBART model...")
        self.tokenizer = MBart50TokenizerFast.from_pretrained(self.model_name)
        self.model = MBartForConditionalGeneration.from_pretrained(self.model_name)
        self.model.to(self.device)

        # Set source and target languages
        self.tokenizer.src_lang = CONFIG["source_lang"]
        self.tokenizer.tgt_lang = CONFIG["target_lang"]

        print("Model loaded successfully!")

    def translate_text(self, text, **kwargs):
        """
        Translate English text to Bengali

        Args:
            text (str): English text to translate
            **kwargs: Additional generation parameters

        Returns:
            str: Bengali translation
        """
        self.model.eval()

        # Set source language for tokenizer
        self.tokenizer.src_lang = CONFIG["source_lang"]

        # Tokenize input text
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            max_length=CONFIG["max_sentence_length"],
            truncation=True,
            padding=True
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Generation parameters
        gen_kwargs = {
            "forced_bos_token_id": self.tokenizer.lang_code_to_id[CONFIG["target_lang"]],
            "max_new_tokens": kwargs.get("max_new_tokens", CONFIG["max_new_tokens"]),
            "num_beams": kwargs.get("num_beams", CONFIG["num_beams"]),
            "repetition_penalty": kwargs.get("repetition_penalty", CONFIG["repetition_penalty"]),
            "length_penalty": kwargs.get("length_penalty", CONFIG["length_penalty"]),
            "temperature": kwargs.get("temperature", CONFIG["temperature"]),
            "do_sample": kwargs.get("do_sample", CONFIG["do_sample"]),
            "top_p": kwargs.get("top_p", CONFIG["top_p"]),
            "early_stopping": True,
            "pad_token_id": self.tokenizer.pad_token_id,
            "eos_token_id": self.tokenizer.eos_token_id
        }

        # Generate translation
        with torch.no_grad():
            generated_tokens = self.model.generate(**inputs, **gen_kwargs)

        # Decode the generated tokens
        translation = self.tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True
        )[0]

        return translation.strip()

    def translate_batch(self, texts, **kwargs):
        """
        Translate a batch of English texts to Bengali

        Args:
            texts (list): List of English texts to translate
            **kwargs: Additional generation parameters

        Returns:
            list: List of Bengali translations
        """
        translations = []

        for text in texts:
            try:
                translation = self.translate_text(text, **kwargs)
                translations.append(translation)
            except Exception as e:
                print(f"Error translating '{text}': {e}")
                translations.append(f"[Translation Error: {text}]")

        return translations

    def interactive_translate(self):
        """
        Interactive translation mode
        """
        print("\n" + "="*60)
        print("Interactive English to Bengali Translation")
        print("Type 'quit' or 'exit' to stop")
        print("="*60)

        while True:
            try:
                english_text = input("\nEnter English text: ").strip()

                if english_text.lower() in ['quit', 'exit', 'q']:
                    print("Goodbye!")
                    break

                if not english_text:
                    print("Please enter some text to translate.")
                    continue

                print("Translating...")
                bengali_translation = self.translate_text(english_text)

                print(f"\nEnglish:  {english_text}")
                print(f"Bengali:  {bengali_translation}")
                print("-" * 60)

            except KeyboardInterrupt:
                print("\nGoodbye!")
                break
            except Exception as e:
                print(f"Error: {e}")

def main():
    """
    Main function to demonstrate the translator
    """
    # Initialize translator
    translator = EnglishToBengaliTranslator()

    # Test sentences
    test_sentences = [
        "Hello, how are you?",
        "I love you.",
        "What is your name?",
        "Good morning.",
        "Thank you very much.",
        "The weather is nice today.",
        "A child in a pink dress is climbing up a set of stairs in an entry way.",
        "A girl going into a wooden building.",
        "A dog is running in the snow.",
        "A man in an orange hat starring at something.",
        "A little girl climbing into a wooden playhouse.",
        "Two dogs of different breeds looking at each other on the road.",
        "I am learning Bengali language.",
        "This is a beautiful flower.",
        "Where is the nearest hospital?",
        "Can you help me?",
        "What time is it?",
        "I would like to eat some food."
    ]

    print("\n" + "="*80)
    print("TRANSLATION RESULTS")
    print("="*80)

    # Translate each sentence
    for i, sentence in enumerate(test_sentences, 1):
        try:
            translation = translator.translate_text(sentence)
            print(f"{i:2d}. English:  {sentence}")
            print(f"    Bengali:  {translation}")
            print("-" * 80)
        except Exception as e:
            print(f"{i:2d}. Error translating: {sentence}")
            print(f"    Error: {e}")
            print("-" * 80)

    # Batch translation example
    print("\n" + "="*80)
    print("BATCH TRANSLATION EXAMPLE")
    print("="*80)

    batch_sentences = [
        "Good morning, how are you today?",
        "I am fine, thank you.",
        "What are you doing?"
    ]

    batch_translations = translator.translate_batch(batch_sentences)

    for eng, ben in zip(batch_sentences, batch_translations):
        print(f"English:  {eng}")
        print(f"Bengali:  {ben}")
        print("-" * 60)

    """
    # Interactive mode (optional)
    print("\n" + "="*80)
    user_input = input("Would you like to try interactive translation? (y/n): ").strip().lower()
    if user_input in ['y', 'yes']:
        translator.interactive_translate()
    """

In [2]:
 main()

Using device: cpu
Loading pre-trained mBART model...


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Model loaded successfully!

TRANSLATION RESULTS
 1. English:  Hello, how are you?
    Bengali:  হাইলো, কেমন আছেন আপনারা?
--------------------------------------------------------------------------------
 2. English:  I love you.
    Bengali:  আমি আপনাদের ভালবাসি ।
--------------------------------------------------------------------------------
 3. English:  What is your name?
    Bengali:  আপনার নাম কি?
--------------------------------------------------------------------------------
 4. English:  Good morning.
    Bengali:  শুভ সকাল ।
--------------------------------------------------------------------------------
 5. English:  Thank you very much.
    Bengali:  অনেক ধন ্ যবাদ ।
--------------------------------------------------------------------------------
 6. English:  The weather is nice today.
    Bengali:  এখন বেশ ভাল ফ ্ যকাল ।
--------------------------------------------------------------------------------
 7. English:  A child in a pink dress is climbing up a set of stairs in a